In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

chat_model = init_chat_model(model="gpt-oss-120b", model_provider="groq")

from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """Get the weather of the given location"""
    return f"The weather is sunny with humidity in {location}"

model_with_tool = chat_model.bind_tools([get_weather]) # bind tool

In [36]:
response = model_with_tool.invoke("What is the weather in Kolkata?")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Kolkata. We need to call function get_weather with location "Kolkata".', 'tool_calls': [{'id': 'fc_0adb93b3-be24-4a32-b2dc-7b459734904a', 'function': {'arguments': '{"location":"Kolkata"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 128, 'total_tokens': 179, 'completion_time': 0.111503041, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.005159368, 'prompt_tokens_details': None, 'queue_time': 0.054782342, 'total_time': 0.116662409}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b7ec5-94e6-7573-9ac1-e31d51dcc364-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Kolkata'}, 'id': 'fc_0adb93b3-be24-4a32-b2dc-7b459734904a', 'type': 'tool_call'

In [39]:
for tool_call in response.tool_calls:
    print(f"Tool name: {tool_call['name']}")
    print(f"Tool args: {tool_call['args']}")
    print(f"location: {tool_call['args']['location']}")

Tool name: get_weather
Tool args: {'location': 'Kolkata'}
location: Kolkata


In [43]:
### Tool execution loops

# Step1: Model generate tool calls
messages = []
messages = [{"role": "user", "content": "What is the weather in Kolkata?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Step2: Tool execution
for tool_call in ai_msg.tool_calls:
    # Execute tool with the arguments
    print("tool call: ", tool_call)
    tool_result = get_weather.invoke(tool_call)
    print("tool result: ", tool_result)
    messages.append(tool_result)

print("messages: ", messages)

# Step3: Model generate final response
final_response = model_with_tool.invoke(messages)
print("AI response: ", final_response)
print("Text: ", final_response.text)


tool call:  {'name': 'get_weather', 'args': {'location': 'Kolkata'}, 'id': 'fc_6ad575db-ce4a-4ae9-9bdd-8a99966508c7', 'type': 'tool_call'}
tool result:  content='The weather is sunny with humidity in Kolkata' name='get_weather' tool_call_id='fc_6ad575db-ce4a-4ae9-9bdd-8a99966508c7'
messages:  [{'role': 'user', 'content': 'What is the weather in Kolkata?'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather in Kolkata?" Need to fetch weather via function get_weather. Use location "Kolkata".', 'tool_calls': [{'id': 'fc_6ad575db-ce4a-4ae9-9bdd-8a99966508c7', 'function': {'arguments': '{"location":"Kolkata"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 128, 'total_tokens': 186, 'completion_time': 0.123737876, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.005687857, 'prompt_tokens_details': None, 'queue_time': 0.056033023, 'total_time': 0